In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

# Smart repo root finder — works regardless of where notebook lives
def is_repo_root(path: Path) -> bool:
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'portfolio_toolkit').exists()

def find_repo_root() -> Path:
    candidates = [Path.cwd()] + list(Path.cwd().parents)
    for path in candidates:
        if is_repo_root(path):
            return path
    raise RuntimeError("Could not find repo root. Make sure you're running from inside the repo.")

repo_root = find_repo_root()
os.chdir(repo_root)
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("repo_root =", repo_root)

from portfolio_toolkit import (
    load_prices,
    build_features,
    validate_feature_frame,
    make_forward_alpha_target,
    slice_split,
    validate_prediction_frame,
    weights_from_predictions_risk_adjusted,
    baseline_weights,
    backtest_weights,
    write_backtest_artifacts,
    build_metrics,
    init_mlflow,
    start_run,
    log_predictions,
    log_portfolio,
    log_backtest,
    log_report_artifacts,
)

print("Imports successful.")

repo_root = C:\Users\brixn\Documents\Portfolio-Optimization-Lib
Imports successful.


In [2]:
# Going to use shared_set_1 for more training
dataset_name = "shared_set_1"
prices = load_prices(dataset_name, repo_root=repo_root)
print("Prices loaded:", prices.shape)
print("Date range:", prices["date"].min(), "->", prices["date"].max())

Prices loaded: (1463605, 8)
Date range: 2014-01-02 00:00:00 -> 2025-12-31 00:00:00


In [3]:
def add_custom_features(features: pd.DataFrame) -> pd.DataFrame:
    """
    Custom notebook-local features built on top of toolkit features.
    All features use only past data — no future information leakage.
    """
    # Risk-adjusted momentum: how strong is momentum relative to volatility
    features["mom_vol_ratio"] = (
        features["momentum_20d"] / features["vol_20d"].replace(0, np.nan)
    )
    # Downside risk ratio: what fraction of total vol is on the downside
    features["downside_ratio"] = (
        features["downside_vol_20d"] / features["vol_20d"].replace(0, np.nan)
    )
    # Momentum divergence: short term vs long term momentum shift
    features["mom_divergence"] = (
        features["momentum_5d"] - features["momentum_20d"]
    )
    # Beta-adjusted momentum: rewards strong returns from low-beta stocks
    features["mom_beta_adjusted"] = (
        features["momentum_20d"] / features["beta_20d_spy"].replace(0, np.nan)
    )
    # Trend strength: combines price trend and relative performance
    features["trend_strength"] = (
        features["price_to_sma_20d"] * features["momentum_20d"]
    )
    return features

# Build 12 toolkit features
features = build_features(prices, feature_names=[
    "momentum_5d",
    "momentum_20d",
    "vol_20d",
    "downside_vol_20d",
    "beta_20d_spy",
    "price_to_sma_20d",
    "rsi_14",
    "macd_hist",
    "bollinger_z_20d",
    "volume_zscore_20d",
    "excess_return_5d_vs_spy",
    "excess_return_20d_vs_spy",
])

# Add 5 custom features
features = add_custom_features(features)
features = validate_feature_frame(features)

# Target: forward alpha vs SPY (not raw return)
# Forces model to predict winners vs losers, not just "stocks go up"
target_alpha = make_forward_alpha_target(prices, horizon=5)

dataset = features.merge(target_alpha, on=["date", "ticker"], how="left").dropna()
print("Dataset shape:", dataset.shape)
print("Total features:", len([c for c in dataset.columns if c not in ["date", "ticker", "forward_alpha_5d_vs_spy"]]))

Dataset shape: (1449922, 20)
Total features: 17


In [4]:
train = slice_split(dataset, dataset_name, "train", repo_root=repo_root)
val   = slice_split(dataset, dataset_name, "val",   repo_root=repo_root)
test  = slice_split(dataset, dataset_name, "test",  repo_root=repo_root)
print("Train:", train.shape)
print("Val:  ", val.shape)
print("Test: ", test.shape)

Train: (702834, 20)
Val:   (248187, 20)
Test:  (498901, 20)


In [5]:
feature_cols = [
    "momentum_5d", "momentum_20d", "vol_20d", "downside_vol_20d",
    "beta_20d_spy", "price_to_sma_20d", "rsi_14", "macd_hist",
    "bollinger_z_20d", "volume_zscore_20d",
    "excess_return_5d_vs_spy", "excess_return_20d_vs_spy",
    "mom_vol_ratio", "downside_ratio", "mom_divergence",
    "mom_beta_adjusted", "trend_strength",
]

target_col = "forward_alpha_5d_vs_spy"

X_train, y_train = train[feature_cols].fillna(0.0), train[target_col]
X_val,   y_val   = val[feature_cols].fillna(0.0),   val[target_col]
X_test           = test[feature_cols].fillna(0.0)

train_data = lgb.Dataset(X_train, label=y_train)
val_data   = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Tuned params — fixes early stopping at iteration 2
# Lower lr + more leaves + regularization = model actually learns
params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.01,
    "num_leaves": 63,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "verbose": -1,
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=1500,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)

print("Best iteration:", model.best_iteration)

Training until validation scores don't improve for 100 rounds
[100]	valid_0's rmse: 0.0474186
Early stopping, best iteration is:
[68]	valid_0's rmse: 0.0474066
Best iteration: 68


In [6]:
predictions = test[["date", "ticker"]].copy()
predictions["horizon"] = 5
predictions["expected_return"] = model.predict(X_test)

# Using vol_20d instead of forward_realized_vol_5d
# vol_20d is calculated from past data only — no future information leakage
predictions["expected_volatility"] = test["vol_20d"]

# Remove SPY — it's the benchmark, not a stock to allocate to
predictions = predictions[predictions["ticker"] != "SPY"].copy()
X_test_no_spy = X_test[test["ticker"].values != "SPY"]
predictions["expected_return"] = model.predict(X_test_no_spy)
predictions["expected_volatility"] = test.loc[test["ticker"] != "SPY", "vol_20d"].values

predictions = validate_prediction_frame(
    predictions, dataset_name=dataset_name, horizon=5, repo_root=repo_root
)
print("Predictions validated:", predictions.shape)
print("Prediction range: min={:.4f}, max={:.4f}".format(
    predictions["expected_return"].min(),
    predictions["expected_return"].max()
))
print("Negative predictions:", (predictions["expected_return"] < 0).sum())
print("Positive predictions:", (predictions["expected_return"] > 0).sum())

Predictions validated: (497903, 5)
Prediction range: min=-0.0315, max=0.0875
Negative predictions: 9491
Positive predictions: 488412


In [8]:
portfolio = weights_from_predictions_risk_adjusted(predictions, dataset_name=dataset_name)
result = backtest_weights(dataset_name, portfolio, repo_root=repo_root)
artifacts = write_backtest_artifacts(result, "runs/Brian_lgbm_v2_alpha")
print("Sharpe:        ", result.metrics.get("sharpe"))
print("Annual Return: ", result.metrics.get("annual_return"))
print("Max Drawdown:  ", result.metrics.get("max_drawdown"))

Sharpe:         0.44662024909446185
Annual Return:  0.08766902865320336
Max Drawdown:   -0.23077692048097953


In [9]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)

print("Feature importance by gain:")
print(importance.to_string(index=False))

Feature importance by gain:
                 feature  importance
          downside_ratio   36.596870
        downside_vol_20d   32.977811
               macd_hist   30.232070
       mom_beta_adjusted   29.134693
            beta_20d_spy   28.218668
                 vol_20d   23.103568
                  rsi_14   17.216262
           mom_vol_ratio   16.642642
excess_return_20d_vs_spy   12.517593
          trend_strength   11.729874
          mom_divergence   11.534138
         bollinger_z_20d   10.478834
       volume_zscore_20d   10.456205
             momentum_5d    8.421704
 excess_return_5d_vs_spy    7.845103
            momentum_20d    6.376996
        price_to_sma_20d    5.986037


In [10]:
def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    """
    Builds the full feature set from raw prices.
    Call this function to reproduce the exact features used during training.
    """
    features = build_features(prices, feature_names=[
        "momentum_5d", "momentum_20d", "vol_20d", "downside_vol_20d",
        "beta_20d_spy", "price_to_sma_20d", "rsi_14", "macd_hist",
        "bollinger_z_20d", "volume_zscore_20d",
        "excess_return_5d_vs_spy", "excess_return_20d_vs_spy",
    ])
    features = add_custom_features(features)
    return features

def predict_from_prices(model, prices: pd.DataFrame, dates=None, tickers=None) -> pd.DataFrame:
    """
    Generates predictions from raw prices using the trained model.
    Filters by dates and tickers if provided.
    Rebalance frequency: daily.
    """
    features = build_model_features(prices)
    target_alpha = make_forward_alpha_target(prices, horizon=5)
    frame = features.merge(target_alpha, on=["date", "ticker"], how="left").dropna()

    if dates is not None:
        frame = frame[frame["date"].isin(dates)]
    if tickers is not None:
        frame = frame[frame["ticker"].isin(tickers)]

    frame = frame[frame["ticker"] != "SPY"]
    X = frame[feature_cols].fillna(0.0)

    predictions = frame[["date", "ticker"]].copy()
    predictions["horizon"] = 5
    predictions["expected_return"] = model.predict(X)
    predictions["expected_volatility"] = frame["vol_20d"]
    return predictions

print("Submission functions defined.")

Submission functions defined.


In [11]:
import json

# Save LightGBM model weights
model_path = repo_root / "MODELS" / "Brian" / "lgbm_v2_model.txt"
model.save_model(str(model_path))

# Save model metadata
metadata = {
    "model_type": "lightgbm",
    "target": "forward_alpha_5d_vs_spy",
    "horizon": 5,
    "rebalance_frequency": "daily",
    "dataset": dataset_name,
    "features": feature_cols,
    "best_iteration": model.best_iteration,
    "params": params,
    "source_notebook": "MODELS/Brian/lgbm_v2_alpha.ipynb",
}
metadata_path = repo_root / "MODELS" / "Brian" / "lgbm_v2_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("Model saved to:", model_path)
print("Metadata saved to:", metadata_path)

Model saved to: C:\Users\brixn\Documents\Portfolio-Optimization-Lib\MODELS\Brian\lgbm_v2_model.txt
Metadata saved to: C:\Users\brixn\Documents\Portfolio-Optimization-Lib\MODELS\Brian\lgbm_v2_metadata.json


In [13]:
import mlflow
from portfolio_toolkit import log_model_submission

init_mlflow(repo_root=repo_root)

with start_run(
    run_name="Brian_lgbm_set1_alpha_v2",
    dataset_name="shared_set_2",
    tags={
        "model_type": "lightgbm",
        "horizon": "5",
        "target": "forward_alpha_vs_spy",
        "actual_dataset": "shared_set_1",
        "version": "2",
        "rebalance_frequency": "daily",
    },
    repo_root=repo_root,
):
    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)
    log_report_artifacts(artifacts)
    log_model_submission(
        model_artifacts={"lgbm_model": model_path},
        model_name="Brian_lgbm_v2_alpha",
        model_family="lightgbm",
        feature_names=feature_cols,
        target="forward_alpha_5d_vs_spy",
        horizon=5,
        rebalance_frequency="daily",
        preprocessing={"fillna": 0.0, "spy_removed": True},
        model_config=params,
        source_files=[repo_root / "MODELS" / "Brian" / "brian_lgbm_v2_alpha.ipynb"],
        notes="v2: alpha target, shared_set_1, 17 features, leakage fixed",
    )

print("Logged to MLflow successfully!")

🏃 View run Brian_lgbm_set1_alpha_v2 at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/1/runs/63b2e00a0c0c4875a0922e7b982a2fdf
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/1
Logged to MLflow successfully!
